In [1]:
# Required Python packages
from datetime import date, datetime, timedelta
import calendar
import pandas as pd
import numpy as np
import itertools
import copy
from numpy.linalg import multi_dot
from scipy.stats import norm
from scipy.stats import bernoulli
import keras
from keras.models import Sequential
from keras.layers import Dense
from keras import layers
from keras.callbacks import EarlyStopping
import keras.optimizers as opt
from scipy.optimize import fmin_slsqp as min
from keras import backend as keras_backend
import matplotlib.pyplot as plt
import math





Using TensorFlow backend.
/opt/tljh/user/lib/python3.6/site-packages/tensorboard/compat/tensorflow_stub/dtypes.py:541: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
/opt/tljh/user/lib/python3.6/site-packages/tensorboard/compat/tensorflow_stub/dtypes.py:542: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
/opt/tljh/user/lib/python3.6/site-packages/tensorboard/compat/tensorflow_stub/dtypes.py:543: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
/opt/tljh/user/lib/python3.6/site-packages/tensorboard/comp

In [251]:
# #Common functions for all programs 

# #The following function finds the date corresponding to the last thursday of a month in a year
# #Input: Year, month - both integers
# #Output: Python date of last thursday of the month
# def find_last_thurs_date_of_month(year, month):
#     calendar_of_the_month = calendar.monthcalendar(year, month)
#     no_of_weeks = len(calendar_of_the_month)
#     if (calendar_of_the_month[no_of_weeks-1][3] != 0):
#         return date(year, month, calendar_of_the_month[no_of_weeks-1][3])
#     else:
#         return date(year, month, calendar_of_the_month[no_of_weeks-2][3])

# #Function for getting previous weekday of a date
# #Input: any python date
# #Output: Previous weekday python date
# def prev_weekday(date):
#     if (date.weekday() == 0):
#         return date - timedelta(days=3)
#     elif (date.weekday() == 6):
#         return date - timedelta(days=2)
#     else:
#         return date - timedelta(days=1)

# #Function to Check for holidays- if holiday, the closing/expiry day is the previous working_day
# #if holiday, select next working day
# #Input: python date, holiday python list
# #Output: Previous working python 
# def prev_workday_if_holiday(date, holidays_list):
#     if (date in holidays_list):
#         return prev_workday_if_holiday(prev_weekday(date), holidays_list)
#     else:
#         return date

# # The function helps in finding the list option last thursday expiries of the month 
# #(we are interest to hedge options of this expiry)
# # Input: Dictionary with keys as years and each key has value of list of months we are interested
# # Output: List of monthly last thursday expiries
# def get_all_monthly_option_expiries(periods_of_interest, holidays_list):
#     monthly_expiries = []
#     for key in periods_of_interest.keys():
#         list_of_months = periods_of_interest[key]
#         expiry_dates = [find_last_thurs_date_of_month(key, month) for month in list_of_months]
#         expiry_dates = [prev_workday_if_holiday(date, holidays_list) for date in expiry_dates]
#         monthly_expiries = monthly_expiries + expiry_dates
#     return monthly_expiries

# #The following function finds the date corresponding to the previous thursdays
# #Input: Date - Python date format, No.of. thursdays back
# #Output: The interested thursday python date
# def date_of_prev_thurs(date, no_of_thurs_back):
#     if (date.weekday() > 3):
#         minus_days = date.weekday() - 3
#         prev_thurs = date - timedelta(days=minus_days) -timedelta(days = 7 * (no_of_thurs_back-1))
#         return prev_thurs
#     else:
#         prev_thurs = date - timedelta(days=no_of_thurs_back * 7)
#         add_days = 3-prev_thurs.weekday()
#         return (prev_thurs + timedelta(days=add_days))

# #Function to find all weekly expiry option dates for every month of interest (i.e. every thursdays)
# #Input: List python monthly expiries
# #Output: dictionary - {'monthly_expiry_date:': [4weeks before thurs, 3w, 2w, 1w]}
# def find_wkly_expries(mthly_expiries, holidays_list):
#     weekly_expiries_dict = {}
#     for expiry_date in mthly_expiries:
#         wkly_expiries_list = [date_of_prev_thurs(expiry_date, no_of_thurs) for no_of_thurs in range(4, 0, -1)]
#         wkly_expiries_list = [prev_workday_if_holiday(date, holidays_list) for date in wkly_expiries_list]
#         weekly_expiries_dict[expiry_date.strftime("%d-%b-%Y")] = wkly_expiries_list
#     return weekly_expiries_dict

# # Funtion to get monthly data_frame names stored for each mkt_data from expiry_date
# #Input: date, product type is "CE", "PE" or "FUT"
# #Output: dataframe string name
# def get_mthly_df_name_from_expiry(expiry_date, prod_type):
#     start_date = date_of_prev_thurs(expiry_date, no_of_thurs_back=4)
#     if (prod_type == "FUT"):
#         df_name_str = "df_" + prod_type + "IDX_BANKNIFTY_" \
#                     + start_date.strftime("%d-%b-%Y") +  "_TO_" \
#                     + expiry_date.strftime("%d-%b-%Y")
#     else:
#             df_name_str = "df_" + "OPTIDX_BANKNIFTY_" + prod_type + "_" \
#                         + start_date.strftime("%d-%b-%Y") +  "_TO_" \
#                         + expiry_date.strftime("%d-%b-%Y")      
#     return df_name_str

# #Load dataframe from csv file of the dats for that month:
# #Input:
# #     Expiry dates list of options to be hedged in python date format
# #     prod_types of the expiries = "CE", "PE", "FUT"
# #Output:
# #      dataframe with keys as same name as csv file
# def load_all_mthly_data(monthly_expiries_list, prod_type_lists=["FUT", "CE", "PE"], relative_path = "mkt_data_covid_region/"):
#     df_dict = dict()
#     for expiry_prod in list(itertools.product(monthly_expiries_list, prod_type_lists)):
#         start_date = date_of_prev_thurs(expiry_prod[0], no_of_thurs_back=4)
#         if (expiry_prod[1] == "FUT"):
#             file_name = expiry_prod[1] + "IDX_BANKNIFTY_" \
#                         + start_date.strftime("%d-%b-%Y") +  "_TO_" \
#                         + expiry_prod[0].strftime("%d-%b-%Y") + ".csv"
#         else:
#             file_name = "OPTIDX_BANKNIFTY_" + expiry_prod[1] + "_" \
#                         + start_date.strftime("%d-%b-%Y") +  "_TO_" \
#                         + expiry_prod[0].strftime("%d-%b-%Y") + ".csv"
#         df_dict["df" + "_" + file_name[:-4]] = pd.read_csv(relative_path + file_name)
#     return (df_dict)


# #Function to find weekly short-term strikes
# def generate_weekly_strikes(wkly_expiries_dict_each_month, relative_path = "mkt_data_covid_region/"):
#     dict_wkly_CE_strikes = {}
#     dict_wkly_PE_strikes = {}
#     dict_wkly_spots = {}
#     for expiry_date_str in wkly_expiries_dict_each_month.keys():
#         ce_strike_list = []
#         pe_strike_list = []
#         spot_list = []
#         for i in range(0, 4):
#             if (i < 3):
#                 ce_filename = "OPTIDX_BANKNIFTY_" + "CE" + "_" \
#                             + wkly_expiries_dict_each_month[expiry_date_str][i].strftime("%d-%b-%Y") +  "_TO_" \
#                             + wkly_expiries_dict_each_month[expiry_date_str][i+1].strftime("%d-%b-%Y") + ".csv"
#                 pe_filename = "OPTIDX_BANKNIFTY_" + "PE" + "_" \
#                             + wkly_expiries_dict_each_month[expiry_date_str][i].strftime("%d-%b-%Y") +  "_TO_" \
#                             + wkly_expiries_dict_each_month[expiry_date_str][i+1].strftime("%d-%b-%Y") + ".csv"
#             else:
#                 ce_filename = "OPTIDX_BANKNIFTY_" + "CE" + "_" \
#                             + wkly_expiries_dict_each_month[expiry_date_str][i].strftime("%d-%b-%Y") +  "_TO_" \
#                             + expiry_date_str + ".csv"
#                 pe_filename = "OPTIDX_BANKNIFTY_" + "PE" + "_" \
#                             + wkly_expiries_dict_each_month[expiry_date_str][i].strftime("%d-%b-%Y") +  "_TO_" \
#                             + expiry_date_str + ".csv"

#             df_ce = pd.read_csv(relative_path + ce_filename)
#             df_ce = df_ce[df_ce['Date'] == wkly_expiries_dict_each_month[expiry_date_str][i].strftime("%d-%b-%Y")]
#             ce_strikes = df_ce['Strike Price'].tolist()
#             df_pe = pd.read_csv(relative_path + pe_filename)
#             df_pe = df_pe[df_pe['Date'] == wkly_expiries_dict_each_month[expiry_date_str][i].strftime("%d-%b-%Y")]
#             pe_strikes = df_pe['Strike Price'].tolist()
#             spot = df_ce['Underlying Value'][0]
            
#             ce_strike_list.append(ce_strikes)
#             pe_strike_list.append(pe_strikes)
#             spot_list.append(spot)
            
#         dict_wkly_CE_strikes[expiry_date_str] = ce_strike_list
#         dict_wkly_PE_strikes[expiry_date_str] = pe_strike_list
#         dict_wkly_spots[expiry_date_str] = spot_list
#     return dict_wkly_CE_strikes, dict_wkly_PE_strikes, dict_wkly_spots

# def generate_mthly_strikes(mthly_mkt_data, prod_moneyness, prod_type, holidays_list):
#     dict_mthly_strikes = {}
#     for data in mthly_mkt_data.keys():
#         if(data[3:6] != 'FUT' and data[20:22] == prod_type):
#             expiry_date = datetime.strptime(mthly_mkt_data[data]['Expiry'][0], "%d-%b-%Y").date()
#             start_date = date_of_prev_thurs(expiry_date, no_of_thurs_back=4)
#             start_date = prev_workday_if_holiday(start_date, holidays_list)
#             file_name = "df_OPTIDX_BANKNIFTY_" + prod_type + "_" \
#                             + start_date.strftime("%d-%b-%Y") +  "_TO_" \
#                             + expiry_date.strftime("%d-%b-%Y") + ".csv"
#             df_month = mthly_mkt_data[file_name[:-4]]
#             df_month = df_month[df_month['Date'] == start_date.strftime("%d-%b-%Y")]
            
#             # Wht percentage to consider as OTM and ITM to be changed later
#             if(prod_moneyness == 'ITM'):
#                 df_month['sort_ind'] = np.abs(np.divide(df_month['Underlying Value'], df_month['Strike Price']) - 1.02)
#             elif(prod_moneyness == 'OTM'):
#                 df_month['sort_ind'] = np.abs(np.divide(df_month['Underlying Value'], df_month['Strike Price']) - 0.98)
#             else:
#                 df_month['sort_ind'] = np.abs(np.divide(df_month['Underlying Value'], df_month['Strike Price']) - 1)
            
#             df_month = df_month.sort_values(by = ['sort_ind'])           
#             dict_mthly_strikes[expiry_date.strftime("%d-%b-%Y")] = df_month['Strike Price'].iloc[0]
    
#     return dict_mthly_strikes
    

# #Function for interpolation of iv - Linear Interpolation
# #Other standard ways of interpolation could be tried in future
# def gen_interpolated_iv(df_iv_surface, vol_date_str, moneyness):
#     df_surf = df_iv_surface[df_iv_surface['Date']==vol_date_str]
#     k_by_s = np.array(df_surf['Moneyness'])
#     imp_vol = np.array(df_surf['Impl_Vol'])
#     if (moneyness <= k_by_s[0]):
#         return imp_vol[0]
#     elif (moneyness >= k_by_s[-1]):
#         return imp_vol[-1]
#     curr_moneyness = k_by_s[0]
#     ind = 0
#     while (moneyness >= curr_moneyness):
#         ind = ind + 1
#         curr_moneyness = k_by_s[ind]
#     interpolated_value = imp_vol[ind-1] + ((imp_vol[ind] - imp_vol[ind-1]) / (k_by_s[ind] - k_by_s[ind-1])) * ((moneyness - k_by_s[ind-1]))
#     return interpolated_value

# def generate_weekly_iv(dict_wkly_expiries_each_month, dict_wkly_spots, dict_mthly_strikes, prod_type,relative_path = "mkt_data_covid_region/"):
#     dict_wkly_iv = {}
#     for each_mnth in dict_wkly_expiries_each_month.keys():
#         #Please note vol smile is same for European Call and Put
#         df_daily_iv_data = pd.read_csv(relative_path + "code_outputs/" + "Daily_vol_smile_CE_" + each_mnth + ".csv")
#         if (prod_type == "PE"):
#             df_iv = copy.deepcopy(df_daily_iv_data)
#             df_iv['Moneyness'] = np.reciprocal(df_daily_iv_data['Moneyness'])
#             iv_list = [gen_interpolated_iv(df_iv, (dict_wkly_expiries_each_month[each_mnth][i]).strftime("%d-%b-%Y"), ((dict_mthly_strikes[each_mnth]) / dict_wkly_spots[each_mnth][i])) for i in range(0, 4)]

#         else:
#             df_iv = copy.deepcopy(df_daily_iv_data)
#             iv_list = [gen_interpolated_iv(df_iv, (dict_wkly_expiries_each_month[each_mnth][i]).strftime("%d-%b-%Y"), (dict_wkly_spots[each_mnth][i] / (dict_mthly_strikes[each_mnth]))) for i in range(0, 4)]
        
#         dict_wkly_iv[each_mnth] = iv_list

#     return dict_wkly_iv

# def generate_weekly_ir(dict_wkly_expiries_each_month, mthly_mkt_data, relative_path = "mkt_data_covid_region/"):
#     dict_wkly_ir = {}
#     for each_month in dict_wkly_expiries_each_month.keys():
#         month_date = datetime.strptime(each_month, "%d-%b-%Y").date()
#         start_day = dict_wkly_expiries_each_month[each_month][0].strftime("%d-%b-%Y")
#         df_fut = mthly_mkt_data['df_FUTIDX_BANKNIFTY_' + start_day + '_TO_' + each_month]
#         weekly_ir_list = []
#         for week in range(0, 4):
#             curr_day = dict_wkly_expiries_each_month[each_month][week]
#             df_week = df_fut[df_fut['Date'] == curr_day.strftime("%d-%b-%Y")]
#             F_t = float(df_week['Settle Price'].tolist()[0])
#             s_t = float(df_week['Underlying Value '].tolist()[0])
#             disc_time = float((month_date - curr_day).days) / 365
# #             r = float(1 / disc_time) * np.log(F_t / s_t)
#             r = 0.03
#             weekly_ir_list.append(r)
#         dict_wkly_ir[each_month] = weekly_ir_list
#     return dict_wkly_ir

# def BSM_call_vec(s_t, k, r, vol, dt):
#     d_1_c = (1 / (vol * (dt ** 0.5))) 
#     d1_log =  (np.log((s_t)/(k)) + ((r + (vol ** 2) / 2) * dt))
#     d_1 = d_1_c * d1_log
#     d_2 = d_1 - vol * (dt ** 0.5)
#     return (np.multiply(norm.cdf(d_1),  s_t) - (norm.cdf(d_2) * k * np.exp(-r * dt)))

# def BSM_put_vec(s_t, k, r, vol, dt):
#     d_1_c = (1 / (vol * (dt ** 0.5))) 
#     d1_log =  (np.log((s_t)/(k)) + ((r + (vol ** 2) / 2) * dt))
#     d_1 = d_1_c * d1_log
#     d_2 = d_1 - vol * (dt ** 0.5)
#     return ((k * np.exp(-r*dt) * norm.cdf(-d_2)) - np.multiply(s_t, norm.cdf(-d_1)))


In [252]:
#Function to price European call options
def pricer_vanilla_by_nn(sim_stock_mat, option_type, opt_strike, dt, \
                         r, vol, nnet_model, batch_size, no_of_epochs,):

    # Current stock price is at index 0. This extracts stock price at the next time step
    stock_vec = sim_stock_mat[:, 1]
        
    # The following steps extract option value/pay-off at the next week
    if (option_type == "CE"):
        option_value = BSM_call_vec(stock_vec, opt_strike, r, vol, dt)
        option_value = np.asarray(option_value)
        option_value = option_value.reshape(-1, 1)
    else:
        option_value = BSM_put_vec(stock_vec, opt_strike, r, vol, dt)
        option_value = np.asarray(option_value)
        option_value = option_value.reshape(-1, 1)
        
    #Creating the training data for the neural network model
    # Neural Network Function using RELU activaion function
    # Build Neural Network model(using Stock prices at next exercise time step and their option value)   
    X_train = stock_vec
    X_train = X_train.reshape(-1, 1)

    Y_train = option_value
    Y_train = np.asarray(Y_train)
    Y_train.reshape(-1, 1)

    nnet_output = nnet_model.fit(X_train, Y_train, epochs=no_of_epochs, batch_size=batch_size, 
                                 verbose=0, validation_split=0.3)

    pfl_weights = np.array(nnet_model.layers[1].get_weights())
            
    return pfl_weights


def pre_train_nn(sim_stock_mat, option_type, \
                 opt_strike, dt, \
                 nnet_model, batch_size, no_of_epochs):

    stock_vec = sim_stock_mat[:, 1]
        
    # The following steps extract option value/pay-off at the next time step
    if (option_type == "CE"):
        option_value = np.maximum(stock_vec - opt_strike, 0)
    else:
        option_value = np.maximum(opt_strike - stock_vec, 0)
        
    option_value = np.asarray(option_value)
    option_value = option_value.reshape(-1, 1)
        
    #Creating the training data for the neural network model
    # Neural Network Function using RELU activaion function
    # Build Neural Network model(using Stock prices at next exercise time step and their option value)
    X_train = stock_vec
    X_train = X_train.reshape(-1, 1)

    Y_train = option_value
    Y_train = np.asarray(Y_train)
    Y_train.reshape(-1, 1, 1)

    nnet_output = nnet_model.fit(X_train, Y_train, epochs=no_of_epochs, batch_size=batch_size, verbose=0, validation_split=0.3)

    return nnet_model


In [253]:
#########################
#User-defined functions #
#########################

#Generate simulations
#Function to generate covariance matrix
def generate_covariance_from_correlation(cor_mat, vol_list, dt):
    vol_diag_mat = np.diag(vol_list)
    cov_mat = np.dot(np.dot(vol_diag_mat, cor_mat), vol_diag_mat) * dt
    return cov_mat

#Generate Simulations of stocks
def gbm1W_simulation(no_of_paths, sim_start_date, sim_end_date, no_of_assets,
                     curr_stock_price, r, vol_list, cor_mat):
    zero_mean = np.zeros(no_of_assets)
    dt = float((sim_end_date - sim_start_date).days) / 365 
    cov_mat = generate_covariance_from_correlation(cor_mat, vol_list, dt)
    
    dw_matx = np.random.multivariate_normal(zero_mean, cov_mat, (int(no_of_paths / 2), 1))
    dw_mat = np.concatenate((dw_matx, -dw_matx), axis=0)
    dw_mat = dw_mat.reshape(no_of_paths, 1)

    sim_ln_stock_mat = np.zeros((no_of_paths, 2))
    sim_ln_stock_mat[:, 0] = np.log(curr_stock_price)
    base_drift = (r - 0.5 * np.square(vol_list[0])) * dt
    curr_drift = sim_ln_stock_mat[:, 0] + base_drift
    sim_ln_stock_mat[:, 1] = curr_drift + dw_mat[:, 0]
    sim_stock_mat = np.exp(sim_ln_stock_mat)
    
    return sim_stock_mat

def generate_weekly_static_pfl_weights(no_of_paths, mthly_expiries_list, \
                                       dict_wkly_expiries_each_month, \
                                       dict_ce_wkly_strikes, dict_pe_wkly_strikes, \
                                       dict_wkly_spots, dict_wkly_iv, \
                                       prod_type, prod_moneyness, option_type = "CE",\
                                       relative_path = "mkt_data_covid_region/", \
                                       no_of_assets=1, cor_mat=[[1]],\
                                       batch_size_divisor=10, no_of_epochs = 1000):
    ce_pfl_weights = []
    pe_pfl_weights = [] 
    cash_weights = []
    for each_month in mthly_expiries_list:
        K = dict_mthly_strikes[each_month.strftime("%d-%b-%Y")]
        weekly_pfl_weights = []
        ce_dict = {}
        pe_dict = {}
        cash_dict = {}
        for week in range(0, 4):
            s_t = dict_wkly_spots[each_month.strftime("%d-%b-%Y")][week]
            vol_list = np.array([dict_wkly_iv[each_month.strftime("%d-%b-%Y")][week]])
            r = np.array([dict_wkly_ir[each_month.strftime("%d-%b-%Y")][week]])
            
            if (week == 3):
                sim_start_date = dict_wkly_expiries_each_month[each_month.strftime("%d-%b-%Y")][week]
                sim_end_date = each_month
            else:
                sim_start_date = dict_wkly_expiries_each_month[each_month.strftime("%d-%b-%Y")][week]
                sim_end_date = dict_wkly_expiries_each_month[each_month.strftime("%d-%b-%Y")][week + 1]
         
            sim_stock_mat = gbm1W_simulation(no_of_paths, sim_start_date, sim_end_date, \
                                             no_of_assets, s_t, r, vol_list, cor_mat)
            
            call_strikes_hdge_list_pos = np.sort(np.array(dict_ce_wkly_strikes[each_month.strftime("%d-%b-%Y")][week])).tolist()
            call_strikes_hdge_list = [-1 * strike for strike in call_strikes_hdge_list_pos]
            put_strikes_hdge_list = np.sort(np.array(dict_pe_wkly_strikes[each_month.strftime("%d-%b-%Y")][week])).tolist()
            no_of_hidden_nodes = len(call_strikes_hdge_list) + len(put_strikes_hdge_list)
            
            call_weights = list(np.ones(len(call_strikes_hdge_list)))
            put_weights = list(-1 * np.ones(len(put_strikes_hdge_list)))
            static_layer_weights = call_weights + put_weights
            static_node_weights = call_strikes_hdge_list + put_strikes_hdge_list
            static_weights = [np.array([static_layer_weights]), np.array(static_node_weights)]

            nnet_model = Sequential()
            hidden_layer = Dense(no_of_hidden_nodes, activation='relu', input_dim=no_of_assets)
            hidden_layer.trainable = False
            nnet_model.add(hidden_layer)
            nnet_model.layers[0].set_weights(static_weights) 
            output_layer = Dense(no_of_output_nodes, activation='linear', kernel_initializer='normal', use_bias=True, kernel_constraint=keras.constraints.NonNeg())
            output_layer.trainable = True
            nnet_model.add(output_layer)
            nnet_model.compile(optimizer=opt.Adam(lr=0.001), loss='mean_squared_error')
            
            dt = float((sim_end_date - sim_start_date).days) / 365
            batch_size = int(no_of_paths/batch_size_divisor)
            nnet_model = pre_train_nn(sim_stock_mat, option_type, \
                                      K, dt, nnet_model, batch_size, no_of_epochs=100)
        
            #Final Params adjustment for the final neural networks
            keras_backend.set_value(nnet_model.optimizer.lr, 5e-4)

            ######################################################################################
            ### Portfolio weights shorter term options from neural network of European options ###
            ######################################################################################

            pfl_weights = pricer_vanilla_by_nn(sim_stock_mat, option_type, K, dt, \
                          r, vol_list[0], nnet_model, batch_size, no_of_epochs)
            
            ce_dates_list = [sim_start_date.strftime("%d-%b-%Y") for i in range(0, len(call_strikes_hdge_list))]
            ce_wt_list = np.array(pfl_weights[0]).reshape(-1)[0:len(call_strikes_hdge_list)]
            ce_dict = {"Date": ce_dates_list, "Strike": call_strikes_hdge_list_pos, "Pfl_weights": ce_wt_list}
            df_ce = pd.DataFrame(ce_dict)
            ce_pfl_weights.append(df_ce)
            
            pe_wt_list = np.array(pfl_weights[0]).reshape(-1)[len(call_strikes_hdge_list):]
            pe_dates_list = [sim_start_date.strftime("%d-%b-%Y") for i in range(0, len(put_strikes_hdge_list))]
            pe_dict = {"Date": pe_dates_list, "Strike":put_strikes_hdge_list, "Pfl_weights": pe_wt_list}
            df_pe = pd.DataFrame(pe_dict)
            pe_pfl_weights.append(df_pe)
            
            cash_date = [sim_start_date.strftime("%d-%b-%Y")]
            cash_weight = np.array(pfl_weights[1][0])
            cash_dict = {"Date":cash_date, "Pfl_weights":cash_weight} 
            df_cash = pd.DataFrame(cash_dict)
            cash_weights.append(df_cash)
            
        df_ce_pfl_weights = pd.concat(ce_pfl_weights, axis=0)
        df_pe_pfl_weights = pd.concat(pe_pfl_weights, axis=0)
        df_cash_pfl_weights = pd.concat(cash_weights, axis=0)
    
        df_ce_pfl_weights.to_csv(relative_path + "code_outputs/B0001_" + prod_moneyness + "_" + "CE_Pfl_weights" + ".csv", index = False)
        df_pe_pfl_weights.to_csv(relative_path + "code_outputs/B0001_" + prod_moneyness + "_" + "PE_Pfl_weights" + ".csv", index = False)
        df_cash_pfl_weights.to_csv(relative_path + "code_outputs/B0001_" +  prod_moneyness + "_" + "Cash_Pfl_weights" + ".csv", index = False)
    
    return df_ce_pfl_weights, df_pe_pfl_weights, df_pe_pfl_weights

def output_strikes_hedged(prod_type, dict_mthly_strikes, relative_path="mkt_data_covid_region/"):
    list_of_dates = []
    list_of_prod_type = []
    list_of_strikes = []
    for each_month in dict_mthly_strikes.keys():
        list_of_dates.append(each_month)
        list_of_strikes.append(dict_mthly_strikes[each_month])
        list_of_prod_type.append(prod_type)
    dict_hedged_prod = {"Date":list_of_dates, "prod_type":list_of_prod_type, "Strike":list_of_strikes}
    df_hedged_prod = pd.DataFrame(dict_hedged_prod) 
    df_hedged_prod.to_csv(relative_path + "code_outputs/" + "B0001_" + prod_type + "_" + "strikes_hedged.csv", index = False)
    return(df_hedged_prod)

In [254]:
###################################
# Common Variables initialisation #
###################################

# The scope of this code is to hedge one option 
# and scope will be extended to a portfolio
no_of_assets = 1
cor_mat = [[1]]

#Path to refer data
source_path = "mkt_data_covid_region/"

#Static hedging performed at different moneyness regions 
#i.e. moneyness is used to select the option with nearest moneyness match
# ATM - At the Money, ITM - In the money, OTM - Out of the Money
prod_moneyness = "ATM"

#Product type to hedge: either "CE" or "PE"
prod_type = "CE"

# Periods of interest will be a dictionary
#Key is the year, value is a list of months 1-12, 1- Jan, 2 - Feb,...12 - Dec
# For E.g., periods_of_interest = {2020: [3], 2019: [11, 12]}
periods_of_interest = {2020: [3]}

#List of holidays
holidays_list = [date(2020, 3, 10)]

#Simulation Parameters
no_of_paths = 1000
#Initialisation of Neural Network hyperparameters for pre-training network
batch_size_divisor = 10
no_of_epochs = 10 #1000 generally
no_of_output_nodes = no_of_assets
batch_size = int(no_of_paths/batch_size_divisor)

#Find the monthly strike of option from mkt data to find the option to be hedged
#Every month, an option is hedged
mthly_expiries_list = get_all_monthly_option_expiries(periods_of_interest, holidays_list)
dict_wkly_expiries_each_month = find_wkly_expries(mthly_expiries_list, holidays_list)

#Load all monthly mkt data
mthly_mkt_data = load_all_mthly_data(mthly_expiries_list, prod_type_lists=["FUT", "CE", "PE"], relative_path = source_path)

#Load weekly strikes of short term options used for hedging monthly options 
dict_ce_wkly_strikes, dict_pe_wkly_strikes, dict_wkly_spots = generate_weekly_strikes(wkly_expiries_dict_each_month)     

#Load monthly strikes of interest as of required moneyness on the begining of month
dict_mthly_strikes = generate_mthly_strikes(mthly_mkt_data, prod_moneyness, prod_type, holidays_list)

#Load weekly IV for Black-Scholes model input and monte-carlo simulation of stock paths
dict_wkly_iv = generate_weekly_iv(dict_wkly_expiries_each_month, dict_wkly_spots, dict_mthly_strikes, prod_type, relative_path = source_path)

#Load weekly risk free interest rates from futures
#This function to be changed based on updates on risk-free-interest - now kept it as cosntant
dict_wkly_ir = generate_weekly_ir(dict_wkly_expiries_each_month, mthly_mkt_data, relative_path = source_path)

#Load Static portfolio weights built by neural network
ce_pfl_wts, pe_pfl_wts, cash_pfl_wts = generate_weekly_static_pfl_weights(no_of_paths, mthly_expiries_list, dict_wkly_expiries_each_month, \
                                                        dict_ce_wkly_strikes, dict_pe_wkly_strikes, dict_wkly_spots, \
                                                        dict_wkly_iv, prod_type, prod_moneyness) 

df_hedged_prod = output_strikes_hedged(prod_type, dict_mthly_strikes)


/opt/tljh/user/lib/python3.6/site-packages/ipykernel_launcher.py:170: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
